# Tutorial 10 — Promote and serve a model locally

**Goal.** Validate a model hand-off and exercise production-shaped request checks.

**Audience.** Engineers moving an offline score into a service.

**Prerequisites.** Base install; `-E serving` enables the optional HTTP cells.

**Produces.** A feature contract, validation examples, latency evidence, and parity checks.

**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
# ruff: noqa
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
# ruff: noqa
dataset = data.require_dataset()
rows = dataset.rows[:12]
feature_schema = {
    key: type(value).__name__ for key, value in rows[0].items() if key not in {"label", "split"}
}
print({"feature_count": len(feature_schema), "sample_features": list(feature_schema)[:8]})

**Run the core operation**


In [ ]:
# ruff: noqa
from fraudtwin.ml import heuristic_predictions

offline_scores = heuristic_predictions(rows)
print(
    pl.DataFrame([item.model_dump(mode="json") for item in offline_scores])
    .select(["event_id", "fraud_score"])
    .head()
)

**Measure and interpret the result**


In [ ]:
# ruff: noqa
# The service contract is testable without Docker.  This cell is an offline fallback.
valid_request = {
    "event_id": rows[0]["event_id"],
    "prediction_timestamp": str(rows[0]["prediction_time"]),
    "features": {"amount": float(rows[0].get("amount", 0.0))},
}
invalid_requests = [
    {},
    {"features": {"unknown": 1}},
    {"event_id": "bad", "prediction_timestamp": "not-a-time", "features": {}},
]
print({"valid_fields": sorted(valid_request), "invalid_cases": len(invalid_requests)})

**Exercise a parameter or failure mode**


In [ ]:
# ruff: noqa
import time

durations = []
for _ in range(25):
    start = time.perf_counter()
    _ = sum(item.fraud_score for item in offline_scores)
    durations.append((time.perf_counter() - start) * 1000)
latencies = sorted(durations)
print(
    {"p50_ms": latencies[len(latencies) // 2], "p95_ms": latencies[int(len(latencies) * 0.95) - 1]}
)

**Write a compact artifact and fingerprint**


In [ ]:
# ruff: noqa
# Optional FastAPI integration: the notebook remains runnable when the extra is absent.
try:
    from fastapi.testclient import TestClient
    from examples.model_service.app import create_app

    print("serving extra detected; use Tutorial 09 artifact with create_app(...)")
except ImportError:
    print("FastAPI is optional; install poetry install -E serving to run HTTP requests")

**Verify invariants and clean up**


In [ ]:
# ruff: noqa
online = [round(item.fraud_score, 12) for item in offline_scores]
replayed = [round(item.fraud_score, 12) for item in heuristic_predictions(rows)]
assert online == replayed
print(
    {
        "replayed": len(replayed),
        "parity": "ok",
        "fallback": "human_review below confidence threshold",
    }
)

**Optional service integration**


In [ ]:
# ruff: noqa
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

**Review the expected outcome**


In [ ]:
# ruff: noqa
# Optional service cell (not required for the offline tutorial path).
try:
    from fastapi.testclient import TestClient
    from examples.model_service.app import create_app

    print("FastAPI service adapter is available; pass a Tutorial 09 artifact to create_app.")
except ImportError:
    print("Install -E serving to run the HTTP adapter.")